# Batchsimulation mit Python

PEExcel-Szenarien automatisiert berechnen und auswerten

## Überblick

Mit Python lassen sich Szenarien des PEQ-Nachweistools (PEExcel) automatisiert simulieren. Die Berechnung wird dabei weiterhin im PEExcel mit VBA ausgeführt. Python steuert die Szenarienauswahl, startet die Simulation und prüft die exportierten Ergebnisse.

**Voraussetzungen**

- Windows mit Microsoft Excel
- PEExcel-Workbook (`.xlsb`) mit dem VBA-Einstiegspunkt `RunScenariosAutomated`
- Vorbereitete Szenarien in der Excel-Tabelle `Scenarios` auf dem Blatt `SCEN`
- Python mit `pywin32`, `pandas`, `openpyxl` und `matplotlib`

Die benötigten Python-Pakete installieren:

```bash
pip install pywin32 pandas numpy openpyxl matplotlib ipython
```

## 1. Pfade und Szenarien konfigurieren

Alle Dateipfade werden relativ zum aktuellen Python-Arbeitsverzeichnis definiert und anschließend in absolute Pfade umgewandelt.

In [14]:
from pathlib import Path

# An das eigene Projekt anpassen
workbook_path = Path(
    "../excel/development/ka_PEQ_Nachweistool_DEV.xlsb"
).resolve()

export_path = Path("../data/exports/batch_001").resolve()

# Zu simulierende Szenarien
scenarios = ["t1", "t2"]

assert workbook_path.is_file(), f"Workbook fehlt: {workbook_path}"

export_path.mkdir(parents=True, exist_ok=True)

print("Arbeitsverzeichnis:", Path.cwd())
print("Workbook:", workbook_path)
print("Exportordner:", export_path)

Arbeitsverzeichnis: c:\Users\schneids\code\peexcel\notebooks
Workbook: C:\Users\schneids\code\peexcel\excel\development\ka_PEQ_Nachweistool_DEV.xlsb
Exportordner: C:\Users\schneids\code\peexcel\data\exports\batch_001


::: {.callout-note}
`Path.cwd()` zeigt das aktuelle Arbeitsverzeichnis. Relative Pfade beziehen sich darauf, nicht zwingend auf den Speicherort des Notebooks.

Der Exportordner sollte für einen ersten Test leer sein, damit vorhandene Dateien nicht mit neuen Simulationsergebnissen verwechselt werden.
:::

## 2. Verbindung mit Excel herstellen

Python verbindet sich nach Möglichkeit mit einer bereits laufenden Excel-Instanz. Ist keine vorhanden, wird Excel gestartet.

In [15]:
import win32com.client as win32

try:
    excel = win32.GetActiveObject("Excel.Application")
except Exception:
    excel = win32.DispatchEx("Excel.Application")

excel.Visible = True

# Bereits geöffnetes Workbook suchen
wb = next(
    (
        w for w in excel.Workbooks
        if Path(w.FullName).resolve() == workbook_path
    ),
    None
)

if wb is None:
    wb = excel.Workbooks.Open(str(workbook_path))

print("Verbunden:", wb.Name)

Verbunden: ka_PEQ_Nachweistool_DEV.xlsb


Bei mehreren gleichzeitig laufenden Excel-Instanzen kann `GetActiveObject` eine andere Instanz auswählen. In diesem Fall die gewünschte Arbeitsmappe gezielt öffnen beziehungsweise die Excel-Instanzen prüfen.

## 3. Verfügbare Szenarien anzeigen

Die Szenarien sind in der strukturierten Excel-Tabelle `Scenarios` hinterlegt.

In [16]:
table = wb.Worksheets("SCEN").ListObjects("Scenarios")

name_col = table.ListColumns("Scenario Name").DataBodyRange
sim_col = table.ListColumns("Simulieren?").DataBodyRange
export_col = table.ListColumns(
    "Export as single project file incl. Timeseries?"
).DataBodyRange

available = [
    name_col.Cells(i, 1).Value
    for i in range(1, table.ListRows.Count + 1)
]

print(f"{len(available)} Szenarien verfügbar:")
print(available[:20])

missing = set(scenarios) - set(available)
assert not missing, f"Unbekannte Szenarien: {missing}"

939 Szenarien verfügbar:
['t1', 't2', 'Aspern PVCheck | F3', 'Aspern PVCheck | F5', 'Aspern PVCheck | F6', 'Aspern PVCheck | F7', 'Aspern PVCheck | F9B', 'Aspern PVCheck | F10', 'Aspern PVCheck | F12', 'Aspern PVCheck | F13', 'Aspern PVCheck | J6B', 'Aspern PVCheck | E7A', 'Aspern PVCheck | H1', 'Aspern PVCheck | C1A', 'Aspern PVCheck | D18C', 'Aspern PVCheck | C4 BT4+5', 'Aspern PVCheck | G11', 'Aspern PVCheck | F9A', 'Aspern PVCheck | H5', 'Aspern PVCheck | m-F3']


Die Tabelle enthält neben den Szenarionamen die Basis-Szenarien und die anzuwendenden Presets. Diese werden von der VBA-Batchsimulation automatisch verarbeitet.

## 4. Szenarien simulieren

Python aktiviert jeweils ein Szenario, setzt den Zeitreihenexport auf `True` und startet die VBA-Simulation.

Die Ergebnisse werden nach jedem Szenario kontrolliert. Bei einem Fehler wird die Schleife abgebrochen.

In [ ]:
from time import perf_counter
from IPython.display import clear_output

start = perf_counter()
results = []

for i, scenario in enumerate(scenarios, 1):
    # Vorherige Auswahl zurücksetzen
    sim_col.Value = False

    # Szenario auswählen
    for row in range(1, table.ListRows.Count + 1):
        if name_col.Cells(row, 1).Value == scenario:
            sim_col.Cells(row, 1).Value = True
            export_col.Cells(row, 1).Value = True
            break

    # VBA-Batchsimulation starten
    success = excel.Run(
        f"'{wb.Name}'!RunScenariosAutomated",
        True
    )

    # Technische Erfolgskontrolle
    output_file = export_path / f"{scenario}.xlsx"

    if not success:
        raise RuntimeError(f"Simulation fehlgeschlagen: {scenario}")

    if not output_file.is_file():
        raise FileNotFoundError(
            f"Zeitreihenexport fehlt: {output_file}"
        )

    results.append({
        "scenario": scenario,
        "file": str(output_file),
        "status": "completed",
    })

    clear_output(wait=True)
    print(f"Abgeschlossen: {i}/{len(scenarios)}")
    print(f"Szenario: {scenario}")
    print(f"Dauer gesamt: {perf_counter() - start:.1f} s")

com_error: (-2147352567, 'Ausnahmefehler aufgetreten.', (0, None, None, None, 0, -2147352571), 2)

::: {.callout-important}
Der VBA-Aufruf ist synchron: Python wartet, bis die Simulation beendet ist. Die Fortschrittsanzeige wird daher nach jedem Szenario aktualisiert.

Die erzeugten Dateinamen entsprechen den durch VBA bereinigten Szenarionamen (`SanitizeFileName`). Bei Sonderzeichen im Szenarionamen muss der erwartete Dateiname entsprechend angepasst werden.
:::

## 5. Exportierte Zeitreihen prüfen

Jeder Zeitreihenexport enthält unter anderem die Blätter `IN`, `OUT`, `SIM2` und `META`.

Die folgende Prüfung kontrolliert, ob die Zeitreihe 8.760 Stundenwerte enthält und der Netzstrombezug `E_grid` gültige Werte aufweist.

In [ ]:
import pandas as pd
import numpy as np

def validate_export(path):
    sim = pd.read_excel(path, sheet_name="SIM2")

    assert len(sim) == 8760, (
        f"Erwartet: 8760 Stunden, erhalten: {len(sim)}"
    )

    assert "E_grid" in sim.columns, (
        "E_grid fehlt im SIM-Export"
    )

    grid = pd.to_numeric(sim["E_grid"], errors="coerce")

    assert grid.notna().all(), "Ungültige Werte in E_grid"
    assert np.isfinite(grid.to_numpy()).all(), "Nicht-endliche Werte"
    assert (grid >= 0).all(), "Negativer Netzstrombezug"

    return {
        "hours": len(sim),
        "E_grid_sum": grid.sum(),
        "E_grid_max": grid.max(),
    }

checks = []

for result in results:
    metrics = validate_export(result["file"])
    checks.append({
        "scenario": result["scenario"],
        **metrics,
    })

pd.DataFrame(checks)

Ein positiver Netzstrombezug ist bei vielen Gebäudeszenarien zu erwarten, aber nicht universell zwingend. Deshalb wird hier nicht generell `E_grid.sum() > 0` verlangt.

Für Referenzszenarien kann eine zusätzliche Mindestwertprüfung sinnvoll sein.

## 6. Zeitreihen visualisieren

Die exportierten SIM-Zeitreihen können unmittelbar mit Python analysiert werden. Im folgenden Beispiel wird der Netzstrombezug für eine ausgewählte Woche dargestellt.

In [ ]:
import matplotlib.pyplot as plt

scenario = scenarios[0]
file = next(
    r["file"] for r in results
    if r["scenario"] == scenario
)

sim = pd.read_excel(file, sheet_name="SIM2")

# Beispiel: Stunden 1 bis 168
week = sim.iloc[:168]

fig, ax = plt.subplots(figsize=(11, 3.5))

ax.plot(
    range(len(week)),
    week["E_grid"],
    label="Netzstrombezug"
)

ax.set(
    title=f"Netzstrombezug – {scenario}",
    xlabel="Stunde",
    ylabel="E_grid (SIM-Einheit)",
)

ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

Die Einheit ist aus dem SIM-Schema zu übernehmen. Für einen vollständigen Strombilanzplot mit PV-Eigennutzung, Netzeinspeisung und Strombedarf siehe [Strombilanz analysieren](https://simonschaluppe.github.io/peexcel/tutorials/howto_analyze_electricity_balance.html).

## 7. Hinweise für größere Simulationsaufträge

Für mehrere hundert oder tausend Szenarien empfiehlt sich eine Aufteilung in kleinere Batches.

- Zunächst mit 2–5 Szenarien testen.
- Exportordner und Dateinamen eindeutig halten.
- Ergebnisse unmittelbar nach jeder Simulation prüfen.
- Fehlgeschlagene Szenarien protokollieren und gezielt wiederholen.
- Bei vielen Szenarien die vorhandene VBA-Option `Skip base reload?` berücksichtigen, um wiederholtes Laden derselben Basis zu vermeiden.

Die gezeigte Python-Schleife führt jedes Szenario einzeln aus. Für größere Aufträge kann dieselbe VBA-Schnittstelle mehrere Szenarien pro Aufruf verarbeiten.

**Weiterführend:** [PEExcel-Dokumentation](https://simonschaluppe.github.io/peexcel/) und [Strombilanzanalyse](https://simonschaluppe.github.io/peexcel/tutorials/howto_analyze_electricity_balance.html).

## Batch mit bereits geöffnetem Excel

`run_batch()` und `inspect_batch()` verwenden eine passende offene Arbeitsmappe der aktiven Excel-Instanz. Diese wird danach weder gespeichert noch geschlossen. Ist keine passende Arbeitsmappe offen, öffnet der Adapter eine eigene Instanz. Eingabeänderungen vor dem Lauf speichern, damit Dateihash und verwendeter Modellstand übereinstimmen.

Für den Python-Batch die öffentliche API statt einer direkten `excel.Run`-Schleife verwenden:

```python
from pexl.batch import run_batch

result = run_batch(
    workbook,
    scenario_filter={"Base Scenario": "Default"},
    output_dir="results/default_batch",
    base_reload="always",
    resume=False,  # bei bestehendem passenden Checkpoint: True
)
```


### Wiederholt neu starten beim Debugging

Mit `run_batch(..., overwrite=True, resume=False)` wird der Ausgabeordner für einen neuen Lauf von bisherigen Batch-Ergebnissen, Checkpoint, Log und Retry-CSV bereinigt. Andere Dateien (etwa `run.py`) und Unterordner bleiben erhalten. Ein vorhandener Run-Lock wird nicht übergangen. Für Wiederaufnahme stattdessen `resume=True` ohne `overwrite` verwenden.

In [ ]:
# Nur zum bewussten Neustart eines Testlaufs aktivieren:
DEBUG_RESTART = False
if DEBUG_RESTART:
    from pexl.batch import run_batch, inspect_batch
    first_ten = inspect_batch(workbook, scenario_filter={"Base Scenario": "Default"}).head(10)
    result = run_batch(workbook, scenarios=first_ten, output_dir="first_10",
                       overwrite=True, resume=False)
